<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Assignment 2: RAG
  </span>
</h2>

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Student Name: Yaron Winter
  </span>
</h2>

In [1]:
print("Install required packages:")
!pip install --upgrade pip

!pip install -q google
!pip install -q google-api-python-client
!pip install -q pypdf
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-openai
!pip install -q sentence-transformers
!pip install -q ragas
print("Done Install!")

Install required packages:
Done Install!


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 1 - Naive Generation
  </span>
</h2>

In [2]:
import pandas as pd
import ast

NEBIUS_TOKENS_FACTORY = "https://api.tokenfactory.nebius.com/v1/"
SORTED_TOP_10_TABLE = "top_10_questions.csv"
SORTED_FULL_TABLE = "sorted_by_id_benchmark.csv"

GENERATED_ANSWER = "generated_answer"
GROUND_TRUTH = "ground_truth"
RAG_ANSWER = "RAG_answer"
VERDICT = "verdict"
JUSTIFY = "justification"
JUDGE = "Judgement"
CORRETNESS = "correctness"
QUESTION_FIELD = "question"
NAIVE_ANSWER = "naive_answer"
EVIDENCE = "evidence"

MAX_RETRIEVED_DOCS = 20

DISPLAY_FIELDS = ["financebench_id", "doc_name", "question_type", "question", "answer", "evidence"]

top_10_df = pd.read_csv(SORTED_TOP_10_TABLE, index_col=0)
full_df = pd.read_csv(SORTED_FULL_TABLE, index_col=0)

In [3]:
print("Tables (after removing metrics-generated + sorting by ID):")
print(f"len(top_10)={len(top_10_df)}, len(full)={len(full_df)}\n")
top_10_df[DISPLAY_FIELDS].head(5)

Tables (after removing metrics-generated + sorting by ID):
len(top_10)=10, len(full)=100



,financebench_id,doc_name,question_type,question,answer,evidence
0,financebench_id_00005,CORNING_2022_10K,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,[{'evidence_text': 'Consolidated Balance Sheet...
1,financebench_id_00070,AMERICANWATERWORKS_2022_10K,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",[{'evidence_text': 'American Water Works Compa...
2,financebench_id_00080,PAYPAL_2022_10K,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"[{'evidence_text': 'PayPal Holdings, Inc.\nCON..."
3,financebench_id_00206,JPMORGAN_2022_10K,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...",[{'evidence_text': 'Overview\nJPMorgan Chase &...
4,financebench_id_00215,VERIZON_2022_10K,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,[{'evidence_text': 'Consolidated Balance Sheet...


In [4]:
# Get Nebius API Key
from getpass import getpass
import os

def get_api_key(name="NEBIUS_API_KEY"):
    api_key = None
    try:
        api_key = os.environ[name]
        print("API Key is taken from the environmental parameters")
    except:
        try:
            api_key = userdata.get(name)
            print("API Key is taken from the google colab user data")
        except:
            try:
                api_key = getpass("Enter API key: ")
                os.environ[name] = api_key
                print("API Key is taken from getpass")
            except:
                raise Exception("API Key for NEBIUS_API_KEY could not be found")

    assert api_key is not None, "API Key is None"
    return api_key

In [5]:
# Allocate AI client
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key = get_api_key(),
)

Enter API key:  ········


API Key is taken from getpass


In [6]:
# Define the LLM response generation code.
import json
def generate_response(attributes: dict,
                      model_name: str,
                      system_prompt: str,
                      user_prompt: str,
                      response_format: dict) -> dict:
    virtual_user_prompt = (user_prompt if attributes is None else f"{user_prompt} {json.dumps(attributes)}```")
    return client.chat.completions.create(
        model=model_name,
        response_format=response_format,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": virtual_user_prompt}
                ],
            },
        ]
    ).to_dict()
print("Done!")

Done!


In [7]:
# Define the prompts for the naive answering generation.
NAIVE_SYSTEM_PROMPT = """
You are an AI agent.
Your task is to answer the questions given to you.
The question to be answered will be presented by a simple JSON-formatted string,
as given in this example:

***Example:***
"{
'Question':'Does Corning have positive working capital based on FY2022 data?...',
}"

***Output Format:***
Please retrive just the generated description, according to the provided format.
"""

NAIVE_USER_PROMPT = """
Please answer this given question.
The question is provided below as JSON-Formatted string between triple backticks:

```json
"""

NAIVE_RESPONSE_SCHEMA = {
  "name": "generated_answer",
  "schema": {
    "type": "object",
    "properties": {
      "generated_answer": {
        "type": "string",
        "description": "An answer to the given question. The length of the answer should not be longer than 90 words"
      },
    },
    "required":["generated_answer"],
    "additionalProperties": False
  },
  "strict": True
}

NAIVE_RESPONSE_FORMAT = {"type": "json_schema", "json_schema": NAIVE_RESPONSE_SCHEMA}

In [8]:
# The code for the naive answers generation.
QUESTION = "Question"

def generate_naive_answer(question: str, model_name: str) -> str:
    q = {QUESTION: question}
    response = generate_response(model_name=model_name,
                                 attributes=q,
                                 user_prompt=NAIVE_USER_PROMPT,
                                 system_prompt=NAIVE_SYSTEM_PROMPT,
                                 response_format=NAIVE_RESPONSE_FORMAT)
    j = ast.literal_eval(response["choices"][0]["message"]["content"])
    return j["generated_answer"]
print("Done!")

Done!


In [11]:
# Generate naive answers to the top 10 entries.
naive_df = top_10_df[["financebench_id", "question_type", QUESTION_FIELD]].copy()
naive_df[GROUND_TRUTH] = top_10_df.answer
naive_df[NAIVE_ANSWER] = naive_df.question.apply(lambda x: generate_naive_answer(x, model_name="meta-llama/Llama-3.3-70B-Instruct"))
naive_df[VERDICT] = "DEFAULT"

In [12]:
# Load the table following the manual analysis.
naive_df = pd.read_csv("assignment2_naive_generation.csv", index_col=0)
naive_df.head(10)

,financebench_id,question_type,question,ground_truth,naive_answer,verdict
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,Corning's FY2022 data shows total current asse...,wrong
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...","According to FY2022 data, American Water Works...",partially correct
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"According to FY2022 data, Paypal's current ass...",wrong
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...","JPM's gross margins are not a relevant metric,...",correct
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,Verizon is considered a capital-intensive busi...,partially correct
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,77.78,Pfizer expects to pay approximately $12 billio...,wrong
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,"Yes, there was a decline of ~42% between FY202...",No data is provided to determine if there was ...,wrong
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,Corporate. Its net revenue was -$473 million.,Corporate segment had the lowest net revenue i...,correct
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,"Yes, change in PPNE was positive year over year","According to Pfizer's financial reports, its P...",partially correct
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,Las Vegas resorts contributed ~90% of company ...,Las Vegas Strip had the highest EBITDAR contri...,correct


### **Analysis & Summary:**

    - There were no cases of refusal, perhaps due to the prompts and response format, which encouraged the LLM to generate an adequately formatted response.

    - The model was pretty confident in all its answers.

    - There were two cases in which the model answered very confidently on a numeric questions, but was completely wrong (lines 0 and 5 in the table); in another case (line 2) it generated response about the relevancy, which was wrong.

    - I could not recognize clear differences between the two question types. In fact, the distributions of the verdicts in both cases are very similar.

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 2 - RAG Reminder
  </span>
</h2>


    - Indexing:
        - A storage of data relevant to the business / application, which can be crucial for generating adequate responses
        - The generation of this storage is done offline, thus spares resources in service time
        - The storage enables the system to remain updated - a convenient way to deal with dynamic or specific data
        - The text chunking might be tricky and induces failures, though
            - too short chunks may not include relevant context, thus may harm the response generation stage
            - too long chunks may reduce the semantic focus of the text, thus may harm the retrieval stage
            
    - Retrieval:
        - This stage is performed online, per query
        - It is used for bringing to the LLM the most relevant information, which is crucial for generating good response
        - It delivers the LLM only a small portion of the data, instead of it all, thus reduce costs and delay time
        - But failures in this stage induces generation failures
            - the LLM does not accept the information needed (relevant docs not retrieved)
            - the LLM accepts too much noise (many docs retrieved, for ensuring high retrieval recall)

    - Generation:
        - Performed online, of course, for each query
        - Does the essential task of the pipeline: generating a response
        - LLMs are particularly strong in analyzing given texts, so using the given docs may contribute a lot
        - But garbage in -> garbage out: the model depends heavily on the docs delivered to it

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 3 - Embed Documents
  </span>
</h2>


In [9]:
# Load the documents and store them in a map, for future usage.
import glob
from pypdf import PdfReader

DOC_NAME = "doc_name"
COMPANY_NAME = "company_name"
COMPANY = "company"
DOC_PERIOD = "doc_period"
PAGE_NUMBER = "page_number"
TEXT = "text"

PDFS_FOLDER = "pdfs/"

pdf_files = glob.glob(PDFS_FOLDER + "*.pdf")
doc_name_to_pdf = {os.path.splitext(os.path.basename(x))[0]:x for x in pdf_files}
print(f"type={type(doc_name_to_pdf)}, len={len(doc_name_to_pdf)}")
print(f"pdf docs:\n{list(doc_name_to_pdf.items())[:3]}")

type=<class 'dict'>, len=42
pdf docs:
[('MGMRESORTS_2022Q4_EARNINGS', 'pdfs/MGMRESORTS_2022Q4_EARNINGS.pdf'), ('PEPSICO_2023Q1_EARNINGS', 'pdfs/PEPSICO_2023Q1_EARNINGS.pdf'), ('PEPSICO_2023_8K_dated-2023-05-05', 'pdfs/PEPSICO_2023_8K_dated-2023-05-05.pdf')]


In [10]:
# Read and parse a pdf document.
def parse_pdf(attributes: dict, pdf_path: str) -> list:
    reader = PdfReader(pdf_path)
    return [
        {
            DOC_NAME: attributes[DOC_NAME],
            COMPANY_NAME: attributes[COMPANY_NAME],
            DOC_PERIOD: attributes[DOC_PERIOD],
            PAGE_NUMBER: i,
            TEXT: page.extract_text(),
        }
        for i, page in enumerate(reader.pages)
    ]

In [11]:
# Split a page into chunks.
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def split_page(page: dict,
               chunk_size=1000,
               chunk_overlap=150):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    base_metadata = {
        k: v for k, v in page.items() if k != TEXT
    }
    
    doc = Document(
        page_content=page[TEXT],
        metadata=base_metadata
    )

    return splitter.split_documents([doc])

In [18]:
# Embed the table's documents.
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from tqdm import tqdm

VECTOR_STORAGE_PATH = "faiss_index"

embedder = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

def embed_table_docs(df: pd.DataFrame, chunk_size=1000, chunk_overlap=150):
    vectorstore = None
    
    handled_docs = set()

    for i in tqdm(range(len(df))):
        doc_name = df.loc[i, "doc_name"]
        if doc_name in handled_docs:
            continue

        handled_docs.add(doc_name)

        try:
            doc_period = df.loc[i, DOC_PERIOD].item()
        except:
            doc_period = df.loc[i, DOC_PERIOD]
            
        attributes = {DOC_NAME: doc_name, DOC_PERIOD: doc_period, COMPANY_NAME: df.loc[i, COMPANY]}
        pages = parse_pdf(attributes=attributes, pdf_path=doc_name_to_pdf[doc_name])

        total_doc_chunks = []
        for page in pages:
            total_doc_chunks.extend(split_page(page=page, chunk_size=chunk_size, chunk_overlap=chunk_overlap))

        if vectorstore is None:
            vectorstore = FAISS.from_documents(total_doc_chunks, embedder)
        else:
            vectorstore.add_documents(total_doc_chunks)

    file_name = f"{VECTOR_STORAGE_PATH}_cs_{chunk_size}_co_{chunk_overlap}"
    vectorstore.save_local(file_name)
print("Done")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done


In [19]:
# Perform the embedding.
embed_table_docs(df=full_df, chunk_size=1000, chunk_overlap=150)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [29:05<00:00, 17.45s/it]


In [20]:
# Load vector dataset function.
def load_vec_dataset(embedder, file_suffix):
    vec_dataset = FAISS.load_local(
        VECTOR_STORAGE_PATH + file_suffix,
        embeddings=embedder,
        allow_dangerous_deserialization=True)
    return vec_dataset

In [21]:
# Perform some tests on the vector storage.
# Define a test /analysis function,for
# checking the retrieval performance
def display_retrieval_results(df: pd.DataFrame, ind: int, k: int, text_len: int, vec_ds):
    try:
        query = df.loc[ind, "question"]
        results = vec_ds.similarity_search(query, k=k)                                        
        print(f"Question: {query}:")
        print(f"\tcompany={df.loc[ind, COMPANY]}")
        print(f"\tdoc={df.loc[ind, DOC_NAME]}")
        print(f"\tqtype={df.loc[ind, "question_type"]}")
        evidence = df.loc[ind, EVIDENCE]
        evidence = ast.literal_eval(evidence)[0]
        print(f"\tevidence: page={evidence["evidence_page_num"]}, text={evidence["evidence_text"][:text_len]}")
        print("\nRetrieved:")
        for i, r in enumerate(results):
            print(f"doc_name[{i}]: {r.metadata[DOC_NAME]}, doc_period: {r.metadata[DOC_PERIOD]}, page: {r.metadata[PAGE_NUMBER]}, text: {r.page_content[:text_len]}")
    except Exception as e:
        print(f"Display Failure:\n\nOriginal Evidence:\n{df.loc[ind, EVIDENCE]}\n\nevidece:\n{evidence}\n\ne:\n{e}")

    return results

In [22]:
vec_ds = load_vec_dataset(embedder=embedder, file_suffix="_cs_1000_co_150")
res = display_retrieval_results(full_df, ind=8, k=3, text_len=50, vec_ds=vec_ds)

Question: Was there any drop in Cash & Cash equivalents between FY 2023 and Q2 of FY2024?:
	company=Best Buy
	doc=BESTBUY_2024Q2_10Q
	qtype=novel-generated
	evidence: page=19, text=July 29, 2023
 
July 30, 2022
 
July 29, 2023
 
Ju

Retrieved:
doc_name[0]: BESTBUY_2023_10K, doc_period: 2023, page: 28, text: January 28, 2023  January 29, 2022
Cash and cash e
doc_name[1]: BESTBUY_2024Q2_10Q, doc_period: 2024, page: 19, text: July 29, 2023  January 28, 2023  July 30, 2022
Cas
doc_name[2]: JOHNSON_JOHNSON_2022_10K, doc_period: 2022, page: 36, text: For discussion related to the fiscal 2022 provisio


### **Retrieval Performance Analysis:**

    - I have tested a few questions from the table, using the above function (display_retrieval_results)
    - In all cases, both the top and the majority of the retrieved chunks came from the correct doc
    - But the page number of the top retrieved chunk was correct only in ~40% of the cases
    - And when the correct page was not at top - it was never among the retrieved chunks, not even for k=15
    - In some cases the evidence text consisted on sequence of numbers
        - such lists of numbers cannot be similar to the textual question, of course

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 4 - RAG Pipeline
  </span>
</h2>

In [23]:
# Define the prompts for the RAG pipeline.
RAG_SYSTEM_PROMPT = """
You are an AI agent who should answer questions based on a given context.
Your task is to analyze the context and find out how it can be used
for generating an adequate response to the question.
Notice that your answer must rely on the given context strictly and solely!

DO NOT USE OTHER KNOWLEDGE SOURCES FOR GENERAING AN ANSWER!

Your answers should be concise and be not longer than 100 words.
You should also cite the documents, based on which you generated the answer.
"""
def build_rag_user_prompt(question: str, results: list) -> tuple:
    """Build the RAG prompt with runtime data enhancement."""
    assert len(results) > 0, "empty retrieved docs list"
    
    contexts = [doc.page_content for doc in results]
    context = "\n\n".join(contexts)

    return f"""Answer the given question based strictly on the provided context
If the context does not contain enough information, just say "No sufficient context is give".
Do not make up information.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:""", context, contexts

In [24]:
# Define the RAG pipeline

ANSWER = "answer"
RETRIEVED_DOCS = "retrieved_chunks"
CONTEXT = "context"

def answer_with_rag(question: str,
                    vec_ds,
                    k: int=4,
                    model_name: str="meta-llama/Llama-3.3-70B-Instruct",
                    reranker=None) -> dict:
    # Build the user prompt.
    num_retrived_docs = (k if reranker is None else MAX_RETRIEVED_DOCS)
    results = vec_ds.similarity_search(query=question, k=num_retrived_docs)

    # Handle the case of empty retrieved docs list.
    if len(results) == 0:
        return {ANSWER: "no retrieved docs", RETRIEVED_DOCS: []}

    # Rerank the retrieved docs list.
    results = (results if reranker is None else reranker(question, results, k))

    # Generate the user prompt.
    user_prompt, context, _ = build_rag_user_prompt(question=question, results=results)

    # Generate the answer.
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=200
    )

    if response.choices[0].message.content is None:
        answer = "NONE"
    else:
        answer = response.choices[0].message.content.strip()
    return {ANSWER: answer, RETRIEVED_DOCS: results, CONTEXT: context}

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 5 - Run & Compare
  </span>
</h2>


In [25]:
# Generate RAG based answers to the first 10 questions.
def get_rag_response(question: str,
                     vec_ds,
                     k: int=4,
                     model_name: str="meta-llama/Llama-3.3-70B-Instruct",
                     reranker=None) -> str:
    d = answer_with_rag(question=question, vec_ds=vec_ds, k=k, model_name=model_name, reranker=reranker)
    return d[ANSWER]

In [27]:
naive_df = pd.read_csv("assignment2_naive_generation.csv", index_col=0)
vec_ds = load_vec_dataset(embedder=embedder, file_suffix="_cs_1000_co_150")
rag_df = naive_df.copy()
rag_df[RAG_ANSWER] = rag_df.question.apply(lambda x: get_rag_response(question=x, vec_ds=vec_ds))
rag_df["Comment"] = "DEFAULT"

In [28]:
# After manual analysis
rag_df = pd.read_csv("assignment2_run_and_compare.csv", index_col=0)
rag_df.head(10)

,financebench_id,question_type,question,ground_truth,naive_answer,verdict,RAG_answer,Comment
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,Corning's FY2022 data shows total current asse...,wrong,No sufficient context is given.\n\nThe provide...,Cannot answer better than wrong?
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...","According to FY2022 data, American Water Works...",partially correct,No sufficient context is given.\n\nThe provide...,naive was better
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"According to FY2022 data, Paypal's current ass...",wrong,No sufficient context is given. The provided c...,Cannot answer better than wrong?
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...","JPM's gross margins are not a relevant metric,...",correct,No sufficient context is given.\n\nThe provide...,naive was better (general knowledge helped)
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,Verizon is considered a capital-intensive busi...,partially correct,"Yes, Verizon is a capital-intensive business. ...",RAG was better! (correct numbers?)
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,77.78,Pfizer expects to pay approximately $12 billio...,wrong,No sufficient context is given.\n\nThe provide...,Cannot answer better than wrong?
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,"Yes, there was a decline of ~42% between FY202...",No data is provided to determine if there was ...,wrong,"Yes, there was a drop. Cash and cash equivalen...",RAG was better! (correct numbers?)
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,Corporate. Its net revenue was -$473 million.,Corporate segment had the lowest net revenue i...,correct,No sufficient context is given.\n\nThe provide...,naive was better:(
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,"Yes, change in PPNE was positive year over year","According to Pfizer's financial reports, its P...",partially correct,No sufficient context is given.\n\nThe provide...,naive was better:(
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,Las Vegas resorts contributed ~90% of company ...,Las Vegas Strip had the highest EBITDAR contri...,correct,Las Vegas Strip Resorts had the highest EBITDA...,both were correct


### **Comparison Discussion:**

    - Overall and clearly: the RAG did not improve the performance, but rather the opposite.
    - In 8 of the 10 questions the RAG responded with "No sufficient context..."
        - in a way it is better than responding with a wrong answer
        - but when it is so frequent - there is a problem
        - need to check deeper, whether the relevant chunks are indeed retrieved
        - but as a lot of the data are long columns of numbers - there might be a generation problen even when relevant chunks are retrieved
    - There is a good point with the RAG, though:
        - in the cases for which it generated answers - the answers were correct!
    - I could not recognize different behavior of the RAG, across the two question types

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 6 - Evaluation
  </span>
</h2>


In [29]:
# Generate answers for the whole dataset
rag_df = full_df[["financebench_id", "question_type", QUESTION_FIELD]].copy()
vec_ds = load_vec_dataset(embedder=embedder, file_suffix="_cs_1000_co_150")
rag_df[GROUND_TRUTH] = full_df.answer
rag_df[RAG_ANSWER] = rag_df.question.apply(lambda x: get_rag_response(question=x, vec_ds=vec_ds))
rag_df.head(5)

,financebench_id,question_type,question,ground_truth,RAG_answer
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,No sufficient context is given. The provided c...
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",No sufficient context is given. The provided c...
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,No sufficient context is given. The provided c...
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...",No sufficient context is given. The provided c...
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,"Yes, Verizon is a capital-intensive business. ..."


In [30]:
rag_df.to_csv("task_6_rag_answers.csv")

In [13]:
# Handle correctness
# Define the correctness judgement prompts
CORRECTNESS_SYS_PROMPT = """
You are an AI agent who provides a judging service.
Your task is to rate a given answer, which was generated
by another AI agent.
The generated answer is a response to a question.
Along with the generated answer you will be given also the ground truth
answer for that question.
The pair of answers will be represented by a JSON-formatted string,
consisting of two keys - ground_truth and generated_answer, as described
in the following example:

*** Query Example ***
"{
'ground_truth': 'Yes, there was a decline of ~42% between FY2023 and Q2 of FY 2024.',
'generated_answer': 'Yes, there was a drop. Cash and cash equivalents decreased from $1,874 in FY 2023 to $1,093 in Q2 of FY2024'
}"

Your task, therefore, is to compare between the two given answers, and rate
the generated answer as either correct or incorrect, based on the similarity
between its content to the content of the ground truth answer.
Here are some guidelines, according to which you should generate your verdict:

*** Example 1 ***:
Query:
{
'ground_truth': 'Yes, there was a decline of ~42% between FY2023 and Q2 of FY 2024.',
'generated_answer': 'Yes, there was a drop. Cash and cash equivalents decreased from $1,874 in FY 2023 to $1,093 in Q2 of FY2024'
}

Response:
{
verdict: correct
justification: the generated answer agrees that there is a decline, and also agree about the percentage of the decline
}

*** Example 2 ***:
Query:
{
'ground_truth': 'Las Vegas resorts contributed ~90% of company level EBITDAR during FY2022.',
'generated_answer': 'Las Vegas Strip Resorts had the highest EBITDAR contribution for MGM during FY2022, with $3.1 billion.'
}

Response:
{
verdict: correct
justification: the agent agrees that Las Vegas had the top EBITDAR contribution
}

*** Example 3 ***:
Query:
{
'ground_truth': 'No, American Water Works had negative working capital of -$1561M in FY 2022.',
'generated_answer': 'No sufficient context is given.\n\nThe provided context does not contain...'
}

Response:
{
verdict: incorrect
justification: when the generated answer starts by "No sufficient context is given" the verdict must be 'incorrect'
}

In general, when the generated answer of the agent agrees with the ground truth answer
about the essense of the issue (e.g. decline or increase, answers to 'who' or 'which', etc.) - 
the verdict should be 'correct', even if the rest of the answers is not particularly similar.
If the essense of the answer is diffrent, or the agent admits that he could not generate
an answer - the verdict is 'incorrect'.

NOTICE: your verdict must be either 'correct' or 'incorrect'!!!
You must retrieve any other value as your verdict!

NOTICE: a generated answer that starts by "No sufficient context is given." must
always be rated as 'incorrect'!!!

***Output Format:***
Please retrive a response according to the provided response_format structure.
Please provide for each criterion a justification, which explains your verdict choice.
"""

CORRECTNESS_USER_PROMPT = """
Please rate the given generated answer based on its similarity to the ground truth answer,
as explained above, and output your decision according to the provided structure.
The two answers are provided below as JSON-Formatted string between triple backticks:

```json
"""

In [14]:
# Define the response format
CORRECTNESS_RESPONSE_SCHEMA = {
  "name": "judge_angent",
  "schema": {
    "type": "object",
    "properties": {
      JUDGE: {
        "type": "object",
        "properties": {
          JUSTIFY: {
            "type": "string",
            "description": "explain your verdict in a single sentence"
          },
          VERDICT: {
            "type": "string",
            "description": "the correctness of the generated answer: correct / incorrect"
          },
        },
        "required": [
          JUSTIFY,
          VERDICT
        ],
        "additionalProperties": False
      },
    },
    "required":[JUDGE],
    "additionalProperties": False
  },
  "strict": True
}

CORRECTNESS_RESPONSE_FORMAT = {"type": "json_schema", "json_schema": CORRECTNESS_RESPONSE_SCHEMA}

In [15]:
# Judge the generated answers correctness.
def rate_answer(r: pd.Series, model_name: str) -> tuple:
    attribute = {GENERATED_ANSWER: r[RAG_ANSWER], GROUND_TRUTH: r[GROUND_TRUTH]}
    response = generate_response(attribute,
                                 model_name,
                                 CORRECTNESS_SYS_PROMPT,
                                 CORRECTNESS_USER_PROMPT,
                                 response_format=CORRECTNESS_RESPONSE_FORMAT
                                )
    try:
        j = ast.literal_eval(response["choices"][0]["message"]["content"])
    except Exception as e:
        raise Exception(f"failed to parse json in rate_answer.\n\ne={e}\n\nresponse={response}\n\natribute={attribute}")
    return (j[JUDGE][VERDICT], j[JUDGE][JUSTIFY],)

In [25]:
# This is the only deepseek model I say in Nebius tokens factory.
# The model stated in the task (DeepSeek-V3-0324) was not there.

def eval_correctness(res_df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    judgements = res_df.apply(lambda r: rate_answer(r, model_name=model_name), axis=1).tolist()
    verdicts = [x[0] for x in judgements]
    res_df["verdict"] = verdicts
    res_df["justify"] = [x[1] for x in judgements]
    return res_df
    
print("Done!")

Done!


In [35]:
model_name = "deepseek-ai/DeepSeek-V3.2"
rag_df = pd.read_csv("task_6_rag_answers.csv", index_col=0)
rag_df = eval_correctness(res_df=rag_df, model_name=model_name)
rag_df.head(25)

,financebench_id,question_type,question,ground_truth,RAG_answer,correctness
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,No sufficient context is given. The provided c...,incorrect
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",No sufficient context is given. The provided c...,incorrect
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,No sufficient context is given. The provided c...,incorrect
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...",No sufficient context is given. The provided c...,incorrect
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,"Yes, Verizon is a capital-intensive business. ...",correct
5,financebench_id_00216,domain-relevant,Does Verizon have a reasonably healthy liquidi...,No. The quick ratio was approximately 0.54 for...,No sufficient context is given. The context do...,incorrect
6,financebench_id_00222,domain-relevant,Does AMD have a reasonably healthy liquidity p...,"Yes. The quick ratio is 1.57, calculated as (c...",No sufficient context is given. The context do...,incorrect
7,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,77.78,"Pfizer expects to pay $1.6 billion, primarily ...",incorrect
8,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,"Yes, there was a decline of ~42% between FY202...",No sufficient context is given to determine th...,incorrect
9,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,Corporate. Its net revenue was -$473 million.,Asset & Wealth Management had the lowest total...,incorrect


In [36]:
rag_df.to_csv("task_6_rag_correctness.csv")

In [37]:
# Handle Faithfullness.
# Notice that we were instructed to use AsyncOpenAI,
# while using the syncronious score computation.
# This cause an error, with no elegant way to resolve it.
# After some digging I found out, that the two optimal
# ways to handle this issue are either making it all
# async, or using evaluate component, for keeping
# things sync.
# I chose, thus, to keep everything sync by using evaluate.
from ragas.metrics import faithfulness
from ragas import evaluate
from langchain_openai import ChatOpenAI
from datasets import Dataset

FAITHFULNESS = "faithfulness"

def compute_faithfulness(res_df: pd.DataFrame, vec_ds, k: int, model_name: str) -> pd.DataFrame:
    # Allocate the llm
    llm = ChatOpenAI(
        model=model_name,
        base_url=NEBIUS_TOKENS_FACTORY,
        api_key=get_api_key(),
    )

    # Prepare dataset
    sub_df = res_df[:20].copy()
    questions = sub_df[QUESTION_FIELD].tolist()
    answers = sub_df[RAG_ANSWER].tolist()
    contexts = []
    for q in questions:
        docs = vec_ds.similarity_search(q, k=k)
        contexts.append([doc.page_content for doc in docs])
    
    dataset = Dataset.from_dict({
            "question": questions,
            "answer": answers,
            "contexts": contexts
        })
    
    results = evaluate(dataset, metrics=[faithfulness], llm=llm)
    eval_df = results.to_pandas()
    faithfulnes_res = eval_df.faithfulness.tolist() + [-1.0]*(len(res_df) - 20)
    res_df[FAITHFULNESS] = faithfulnes_res
    return res_df

/tmp/ipykernel_337944/985798270.py:10: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness


In [38]:
# Handle retrieval hit rate.
def retrieval_hit(df: pd.DataFrame, results: list, ind: int, k: int) -> int:
    try:
        query = df.loc[ind, QUESTION_FIELD]
        evidence = df.loc[ind, EVIDENCE]
        evidence = evidence.replace("\'", "\"")
        evidence = ast.literal_eval(evidence)[0]
        evidence_page_num = evidence["evidence_page_num"]
    except Exception as e:
        #print(f"Failed on index {i}, e={e}")
        return 0

    for r in results[:k+1]:
        if r.metadata['page_number'] == evidence_page_num:
            return 1
    return 0

In [39]:
# Compute retrieval hits & faithfulness.
model_name = "deepseek-ai/DeepSeek-V3.2"
vec_ds = load_vec_dataset(embedder=embedder, file_suffix="_cs_1000_co_150")
rag_df = pd.read_csv("task_6_rag_correctness.csv", index_col=0)
rag_df = compute_faithfulness(res_df=rag_df, vec_ds=vec_ds, k=4, model_name=model_name)
for k in {1,3,5}:
    hits = []
    for i in range(len(full_df)):
        docs = vec_ds.similarity_search(full_df.loc[i, QUESTION_FIELD], k=k)
        hits.append(retrieval_hit(df=full_df, results=docs, ind=i, k=k))
    rag_df[f"page_hit_at_{k}"] = hits

API Key is taken from the environmental parameters


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

In [40]:
rag_df.to_csv("task_6_rag_faithful_n_hr.csv")

In [42]:
# Get average performance:
rag_df = pd.read_csv("task_6_rag_faithful_n_hr.csv", index_col=0)
print(f"Mean Correctness\t{rag_df[CORRETNESS].apply(lambda x: 1 if x=="correct" else 0).mean():.3f}")
print(f"Mean Faithfulness\t{rag_df[:20]["faithfulness"].mean():.3f}")
print(f"Mean hit@1\t{rag_df["page_hit_at_1"].mean():.3f}")
print(f"Mean hit@3\t{rag_df["page_hit_at_3"].mean():.3f}")
print(f"Mean hit@5\t{rag_df["page_hit_at_5"].mean():.3f}")
rag_df.to_csv("assignment2_evaluation.csv")

Mean Correctness	0.260
Mean Faithfulness	0.773
Mean hit@1	0.170
Mean hit@3	0.260
Mean hit@5	0.310


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 7 - Improvement Cycles
  </span>
</h2>


In [43]:
from tqdm import tqdm
def performance_stats(raw_df: pd.DataFrame, res_df: pd.DataFrame, ks: list, vec_ds, reranker):
    for k in ks:
        res_df[f"page_hit_at_{k}"] = [0] * len(res_df)
        
    for i in tqdm(range(len(raw_df))):
        num_top_docs = (k if reranker is None else MAX_RETRIEVED_DOCS)
        q = raw_df.loc[i, QUESTION_FIELD]
        docs = vec_ds.similarity_search(q, k=num_top_docs)
        docs = (docs if reranker is None else reranker(question=q, docs=docs, top_k=k))
        for k in ks:
            res_df.loc[i, f"page_hit_at_{k}"] = retrieval_hit(df=raw_df, results=docs, ind=i, k=k)
    
    print(f"Mean Correctness\t{res_df["correctness"].apply(lambda x: 1 if x=="correct" else 0).mean():.3f}")
    print(f"Mean Faithfulness\t{res_df[:20][FAITHFULNESS].mean():.3f}")
    
    for k in ks: 
        key = f"page_hit_at_{k}"
        name = f"hit@{k}"
        print(f"Mean {name}\t{res_df[key].mean():.3f}")

In [44]:
print("Baseline Results:\n")
vec_ds = load_vec_dataset(embedder=embedder, file_suffix="_cs_1000_co_150")
rag_df = pd.read_csv("assignment2_evaluation.csv", index_col=0)
performance_stats(raw_df=full_df, res_df=rag_df, ks=[1, 4, 10, 15, 20, 25, 30], vec_ds=vec_ds, reranker=None)

Baseline Results:



100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 36.50it/s]

Mean Correctness	0.260
Mean Faithfulness	0.773
Mean hit@1	0.230
Mean hit@4	0.310
Mean hit@10	0.380
Mean hit@15	0.410
Mean hit@20	0.450
Mean hit@25	0.460
Mean hit@30	0.480


In [45]:
# The test function.
def run_test_config(df: pd.DataFrame,
                    k: int,
                    vec_ds_suffix: str,
                    answer_model_name: str,
                    judge_model_name: str,
                    reranker) -> pd.DataFrame:
    print("Load table")
    BASE_FIELDS = ["financebench_id", "question_type", "question", "ground_truth", "experiment", "change"]
    rag_df = df[BASE_FIELDS].copy()

    print("Load vectors storage")
    vec_ds = load_vec_dataset(embedder=embedder, file_suffix=vec_ds_suffix)

    print("Get RAG answers")
    rag_df[RAG_ANSWER] = rag_df.question.apply(
        lambda x: get_rag_response(question=x,
                                   vec_ds=vec_ds,
                                   k=k,
                                   model_name=answer_model_name,
                                   reranker=reranker))

    
    print("Compute Correctness")
    rag_df = eval_correctness(res_df=rag_df, model_name=judge_model_name)

    print("Compute Faithfulness")
    rag_df = compute_faithfulness(res_df=rag_df, vec_ds=vec_ds, k=num_top_docs, model_name=judge_model_name)
    
    return rag_df

### **Experiment 1: Increase the number of retrieved documents**

Hypothesis:

    - For k=4 only 31% of the relevant documents are retrieved.
    - Increasing k to 20 will increase the rate of retrieved relevant documents to 45%.
    - It is a very simple experiment to start with.

In [46]:
rag_df = pd.read_csv("assignment2_evaluation.csv", index_col=0)
answer_model_name = "meta-llama/Llama-3.3-70B-Instruct"
judge_model_name = "deepseek-ai/DeepSeek-V3.2"
num_top_docs = 20
rag_df["experiment"] = "1"
rag_df["change"] = "retrieved docs number"
rag_df = run_test_config(df=rag_df,
                         k=num_top_docs,
                         vec_ds_suffix="_cs_1000_co_150",
                         answer_model_name=answer_model_name,
                         judge_model_name=judge_model_name,
                         reranker=None)
rag_df.to_csv("assignment2_improvement_cycles_increase_num_retrieved.csv")

print("Experiment 1 (increase retrieved docs number) Results:\n")
vec_ds = load_vec_dataset(embedder=embedder, file_suffix="_cs_1000_co_150")
performance_stats(raw_df=full_df, res_df=rag_df, ks=[1,3,5,20], vec_ds=vec_ds, reranker=None)

Load table
Load vectors storage
Get RAG answers
Compute Correctness
Compute Faithfulness
API Key is taken from the environmental parameters


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Experiment 1 (increase retrieved docs number) Results:



100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 38.18it/s]

Mean Correctness	0.380
Mean Faithfulness	0.776
Mean hit@1	0.230
Mean hit@3	0.290
Mean hit@5	0.320
Mean hit@20	0.450


### **Experiment 1: Conclusions**

    - The correctness was improved nicely: 26% -> 38%!
    - The faithfulness remained about the same.
    - Although sending many documents to the LLM may harm generation, in this case it worked very well.

### **Experiment 2: Try different chunks sizes**

Hypothesis:

    - Try the impact of increasing and decreasing of the chunk size.
    - Tuning chunk size may impact both the retrieval quality and the generation stage.

In [48]:
print("Test shorter chunks")
rag_df = pd.read_csv("assignment2_evaluation.csv", index_col=0)
answer_model_name = "meta-llama/Llama-3.3-70B-Instruct"
judge_model_name = "deepseek-ai/DeepSeek-V3.2"
num_top_docs = 4
rag_df["experiment"] = "2"
rag_df["change"] = "shorter chunks"
rag_df = run_test_config(df=rag_df,
                         k=num_top_docs,
                         vec_ds_suffix="_cs_500_co_150_full_doc_text",
                         answer_model_name=answer_model_name,
                         judge_model_name=judge_model_name,
                         reranker=None)

print("Experiment 2 (shorter chunks) Results:\n")
vec_ds = load_vec_dataset(embedder=embedder, file_suffix="_cs_500_co_150_full_doc_text")
performance_stats(raw_df=full_df, res_df=rag_df, ks=[1,3,5,20], vec_ds=vec_ds, reranker=None)
rag_df.to_csv("assignment2_improvement_cycles_shorter_chunks.csv")

Test shorter chunks
Load table
Load vectors storage
Get RAG answers
Compute Correctness
Compute Faithfulness
API Key is taken from the environmental parameters


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Experiment 2 (shorter chunks) Results:



100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 34.04it/s]

Mean Correctness	0.250
Mean Faithfulness	0.653
Mean hit@1	0.220
Mean hit@3	0.290
Mean hit@5	0.310
Mean hit@20	0.470


In [49]:
print("Test longer chunks")
rag_df = pd.read_csv("assignment2_evaluation.csv", index_col=0)
answer_model_name = "meta-llama/Llama-3.3-70B-Instruct"
judge_model_name = "deepseek-ai/DeepSeek-V3.2"
num_top_docs = 4
rag_df["experiment"] = "2"
rag_df["change"] = "longer chunks"
rag_df = run_test_config(df=rag_df,
                         k=num_top_docs,
                         vec_ds_suffix="_cs_1750_co_100_full_doc_text",
                         answer_model_name=answer_model_name,
                         judge_model_name=judge_model_name,
                         reranker=None)

print("Experiment 2 (longer chunks) Results:\n")
vec_ds = load_vec_dataset(embedder=embedder, file_suffix="_cs_1750_co_100_full_doc_text")
performance_stats(raw_df=full_df, res_df=rag_df, ks=[1,3,5,20], vec_ds=vec_ds, reranker=None)
rag_df.to_csv("assignment2_improvement_cycles_longer_chunks.csv")

Test longer chunks
Load table
Load vectors storage
Get RAG answers
Compute Correctness
Compute Faithfulness
API Key is taken from the environmental parameters


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Experiment 2 (longer chunks) Results:



100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 39.70it/s]

Mean Correctness	0.290
Mean Faithfulness	0.768
Mean hit@1	0.170
Mean hit@3	0.230
Mean hit@5	0.320
Mean hit@20	0.460


### **Experiment 2: Conclusions**

    - Both increasing and decreasing the chunks' length did not improve the baseline performance.
    - Besides, hit@k metrics remained pretty much the same as the baseline.
    - Evidently, tuning the chunk's length will not help us, in this case.

### **Experiment 3: Rerank retrieved documents**

Hypothesis:

    - The reranker may improve the top@k results.
    - This way we may benefit both increase in top@k, while keeping the number of docs sent to the llm small.

In [50]:
# Define the reranking function.
from sentence_transformers import CrossEncoder

def rerank_docs(model_name: str="BAAI/bge-reranker-base"):
    reranker = CrossEncoder(model_name)
    def rerank(question: str, docs: list, top_k=4) -> list:
        ques_doc_pairs = [(question, doc.page_content) for doc in docs]

        scores = reranker.predict(ques_doc_pairs)

        ranked = sorted(
            zip(docs, scores),
            key=lambda x: x[1],
            reverse=True
        )

        return [d for d, _ in ranked[:top_k]]
    return rerank

reranker = rerank_docs()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
print("Test Reranker")
rag_df = pd.read_csv("assignment2_evaluation.csv", index_col=0)
answer_model_name = "meta-llama/Llama-3.3-70B-Instruct"
judge_model_name = "deepseek-ai/DeepSeek-V3.2"
num_top_docs = 4
rag_df["experiment"] = "3"
rag_df["change"] = "rerank retrieved docs"
rag_df = run_test_config(df=rag_df,
                         k=num_top_docs,
                         vec_ds_suffix="_cs_1000_co_150",
                         answer_model_name=answer_model_name,
                         judge_model_name=judge_model_name,
                         reranker=reranker)

print("Experiment 2 (reranking docs) Results:\n")
vec_ds = load_vec_dataset(embedder=embedder, file_suffix="_cs_1000_co_150")
performance_stats(raw_df=full_df, res_df=rag_df, ks=[1,4,7,10], vec_ds=vec_ds, reranker=reranker)
rag_df.to_csv("assignment2_improvement_cycles_reranker.csv")

Test Reranker
Load table
Load vectors storage
Get RAG answers
Compute Correctness


<unknown>:3: SyntaxWarning: invalid escape sequence '\/'


Compute Faithfulness
API Key is taken from the environmental parameters


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Experiment 2 (reranking docs) Results:



100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [09:04<00:00,  5.45s/it]

Mean Correctness	0.290
Mean Faithfulness	0.494
Mean hit@1	0.130
Mean hit@4	0.220
Mean hit@7	0.310
Mean hit@10	0.350


### **Experiment 3: Conclusions**

    - The results of the reranking are disappointing:
        - there is a very small improvement in correctness: 26% -> 29% (w.r.t. the baseline)
        - but there is a sharp decrease in faithfulness: 0.773 -> 0.494
        - besides there are clear degradations in all hit@k metrics
    - So, evidently, the reranker does not improve performance at all.
    - Besides, applying the reranker slows down the running time significantly.

### **Experiment 4: Try another generation model**

Hypothesis:

    - For this experiment I use the best retrieval configutaion
        - top_k=20, which gave the best and the most reliable performance.
    - A larger / better model may improve the generation.
    - I will try gpt-oss-120b, google/gemma-3-27b-it, and openai/gpt-oss-120b

In [52]:
print("Test Larger model + top_k=20")
rag_df = pd.read_csv("assignment2_evaluation.csv", index_col=0)
answer_model_name = "Qwen/Qwen3-235B-A22B-Instruct-2507"
judge_model_name = "deepseek-ai/DeepSeek-V3.2"
num_top_docs = 20
rag_df["experiment"] = "4"
rag_df["change"] = "use Qwen/Qwen3-235B-A22B-Instruct-2507"
rag_df = run_test_config(df=rag_df,
                         k=num_top_docs,
                         vec_ds_suffix="_cs_1000_co_150",
                         answer_model_name=answer_model_name,
                         judge_model_name=judge_model_name,
                         reranker=None)

print("Experiment 4 (larger models) Results:\n")
vec_ds = load_vec_dataset(embedder=embedder, file_suffix="_cs_1000_co_150")
performance_stats(raw_df=full_df, res_df=rag_df, ks=[1,3,5,7,10], vec_ds=vec_ds, reranker=None)
rag_df.to_csv("assignment2_improvement_cycles_swen_llm.csv")

Test Larger model + top_k=20
Load table
Load vectors storage
Get RAG answers
Compute Correctness
Compute Faithfulness
API Key is taken from the environmental parameters


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Experiment 4 (larger models) Results:



100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 37.05it/s]

Mean Correctness	0.380
Mean Faithfulness	0.711
Mean hit@1	0.230
Mean hit@3	0.290
Mean hit@5	0.320
Mean hit@7	0.350
Mean hit@10	0.380


### **Experiment 4: Conclusions**

    - Swen was the only model that did not harm performance, w.r.t. the performance of baseline + top_k=20
    - both OpenAI and Gemini models cause degradation in performance.

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Wrap-Up
  </span>
</h2>

    - The retrieval metrics are pretty low: only 45% for top_k=20, so it may be improved significantly
    - We also saw that only change that gave a considerable positive impact was increasing the top_k
        - So the retrieval is definitely a component that may be improved further
    - Three other LLMs that were tested - including these of OpenAI & Gemini - did not help at all
        - this suggests that the generation is not barrier of its own
        - it more about what is given to the LLM, either through the retrieval or the data itself
        - indeed, a lot of the data are lists of numbers, which harden both the retrieval and the generation
    - If I had another week I would:
        - start by deeper quality analysis, such as for example:
            - the percentage of generations that accept relevant document and generate correct answers
            - check carefully how reliable are the automatic rates of correctness and faithfulness
            - some sporadic analysis of the raw data
                - for getting impression whether there is a chance for improvement at all
                - and perhaps for getting some new ideas
        - Try to improve the 'peripheral' components
            - Embedder
            - Reranker


In [19]:
# Generate naive answers to the top 10 entries.
naive_df = full_df[["financebench_id", "question_type", QUESTION_FIELD]].copy()
naive_df[GROUND_TRUTH] = full_df.answer
naive_df[NAIVE_ANSWER] = naive_df.question.apply(lambda x: generate_naive_answer(x, model_name="meta-llama/Llama-3.3-70B-Instruct"))
naive_df[VERDICT] = "DEFAULT"

In [22]:
naive_df.head(10)


,financebench_id,question_type,question,ground_truth,naive_answer,verdict
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,"Based on Corning's FY2022 data, working capita...",DEFAULT
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",American Water Works' FY2022 data shows total ...,DEFAULT
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,Paypal's FY2022 data shows total current asset...,DEFAULT
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...","JPM's gross margins are not a relevant metric,...",DEFAULT
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,"According to FY2022 data, Verizon's capital ex...",DEFAULT
5,financebench_id_00216,domain-relevant,Does Verizon have a reasonably healthy liquidi...,No. The quick ratio was approximately 0.54 for...,Verizon's quick ratio for FY 2022 is not publi...,DEFAULT
6,financebench_id_00222,domain-relevant,Does AMD have a reasonably healthy liquidity p...,"Yes. The quick ratio is 1.57, calculated as (c...","AMD's quick ratio for FY22 is around 1.27, ind...",DEFAULT
7,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,77.78,Pfizer expects to pay approximately $12 billio...,DEFAULT
8,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,"Yes, there was a decline of ~42% between FY202...",No data is provided to determine if there was ...,DEFAULT
9,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,Corporate. Its net revenue was -$473 million.,Corporate segment had the lowest net revenue i...,DEFAULT


In [26]:
RAG_ANSWER = "naive_answer"
CORRECTNESS = "veridct"
model_name = "deepseek-ai/DeepSeek-V3.2"
naive_df = naive_df[naive_df.columns[:-2]].copy()
naive_df = eval_correctness(res_df=naive_df, model_name=model_name)
naive_df.head(25)

,financebench_id,question_type,question,ground_truth,naive_answer,verdict,justify
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,"Based on Corning's FY2022 data, working capita...",incorrect,The generated answer partially agrees with the...
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",American Water Works' FY2022 data shows total ...,incorrect,The ground truth states explicitly that Americ...
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,Paypal's FY2022 data shows total current asset...,correct,The generated answer states that PayPal has po...
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...","JPM's gross margins are not a relevant metric,...",correct,The generated answer correctly identifies the ...
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,"According to FY2022 data, Verizon's capital ex...",incorrect,The comparison of the two texts reveals a disc...
5,financebench_id_00216,domain-relevant,Does Verizon have a reasonably healthy liquidi...,No. The quick ratio was approximately 0.54 for...,Verizon's quick ratio for FY 2022 is not publi...,incorrect,The generated answer fails to directly address...
6,financebench_id_00222,domain-relevant,Does AMD have a reasonably healthy liquidity p...,"Yes. The quick ratio is 1.57, calculated as (c...","AMD's quick ratio for FY22 is around 1.27, ind...",incorrect,Based on the comparison between the Generated ...
7,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,77.78,Pfizer expects to pay approximately $12 billio...,incorrect,The generated answer provides a specific numer...
8,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,"Yes, there was a decline of ~42% between FY202...",No data is provided to determine if there was ...,incorrect,"The generated answer states ""No data is provid..."
9,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,Corporate. Its net revenue was -$473 million.,Corporate segment had the lowest net revenue i...,correct,The generated answer identifies the Corporate ...


In [34]:
ind = 2
print(f"q: {naive_df.loc[ind, "question"]}")
print(f"gt: {naive_df.loc[ind, "ground_truth"]}")
print(f"na: {naive_df.loc[ind, "naive_answer"]}")
print(f"v: {naive_df.loc[ind, "verdict"]}")
print(f"j: {naive_df.loc[ind, "justify"]}")

q: Does Paypal have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.
gt: Yes. Paypal has a positive working capital of $ 1.6Bn as of FY2022 end.
na: Paypal's FY2022 data shows total current assets of $76.7B and total current liabilities of $26.8B, resulting in positive working capital of $49.9B. This indicates Paypal's ability to meet short-term obligations.
v: correct
j: The generated answer states that PayPal has positive working capital in FY2022, which aligns with the ground truth that PayPal has positive working capital for FY2022. However, there is a discrepancy in the specific numerical values, and the generated answer includes extensive additional details. Despite minor numerical variances, the overall positive working capital conclusion remains, so the generated answer is consistent in agreement with the affirmation of positive working capital in FY2022.


In [36]:
print(len(naive_df[naive_df.verdict=="correct"]))

43
